In [ ]:
# Scenario: AI-Powered Study Assistant (Flashcard-Based)
# 1. State Definition
# The assistant maintains a notebook-like state for each learner:
# - topic → The subject the learner is studying (e.g., "Photosynthesis").
# - flashcard → A generated Q&A card created by Groq (question on one side, answer on the other).
# - progress → A log of all past flashcards attempted, including whether the learner got them correct or not.

# 2. Workflow (Graph of States)
# Each learner interaction flows through nodes until a flashcard is delivered:
# - Input Node
# - Learner provides a topic or asks for practice (e.g., "Test me on cell biology").
# - State updates: topic = "cell biology"
# - Generation Node (Groq API)
# - Groq generates a flashcard:
# - flashcard.question = "What organelle is known as the powerhouse of the cell?"
# - flashcard.answer = "Mitochondria"
# - Response Node
# - Assistant presents the flashcard question to the learner.
# - Evaluation Node
# - Learner responds with their answer.
# - Assistant checks correctness and updates progress.
# - History Node
# - Logs the flashcard attempt:
# - progress = [{question, learner_answer, correct/incorrect}]

In [4]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict
import requests
import os
from dotenv import load_dotenv

load_dotenv()

# 1. State Definition
class StudyState(TypedDict):
    topic: str
    flashcard: Dict[str, str]
    user_answer: str
    feedback: str
    progress: List[Dict]

# 2. Generate Flashcard
import re
import requests
import os

def generate_flashcard(state):
    api_key = os.getenv("NVIDIA_API_KEY")

    prompt = f"""
Generate ONE flashcard for the topic: {state['topic']}

STRICT FORMAT:
Question: <one clear question>
Answer: <one clear short answer>

Do NOT add anything else.
"""

    response = requests.post(
        "https://integrate.api.nvidia.com/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        },
        json={
            "model": "meta/llama3-8b-instruct",
            "messages": [
                {"role": "system", "content": "You are a study assistant."},
                {"role": "user", "content": prompt}
            ],
            "temperature": 0.3
        }
    )

    data = response.json()
    content = data["choices"][0]["message"]["content"]

    print("\nRAW LLM OUTPUT:\n", content)  # debug

    # Regex parsing
    match_q = re.search(r"Question:\s*(.*)", content)
    match_a = re.search(r"Answer:\s*(.*)", content)

    q = match_q.group(1).strip() if match_q else None
    a = match_a.group(1).strip() if match_a else None

    # fallback (IMPORTANT)
    if not q or not a:
        q = f"What is a key concept in {state['topic']}?"
        a = "Concept explanation unavailable"

    return {
        "flashcard": {
            "question": q,
            "answer": a
        }
    }

# 3. Ask Question
def ask_question(state: StudyState):
    print("\n📚 Question:", state["flashcard"]["question"])
    user_ans = input("Your Answer: ")

    return {"user_answer": user_ans}

# 4. Evaluate Answer
def evaluate_answer(state):
    correct_ans = state["flashcard"]["answer"].strip().lower()
    user_ans = state["user_answer"].strip().lower()

    if not correct_ans:
        is_correct = False
    else:
        is_correct = correct_ans in user_ans

    feedback = "✅ Correct!" if is_correct else f"❌ Incorrect. Correct answer: {correct_ans}"

    return {
        "feedback": feedback,
        "progress": state["progress"] + [{
            "question": state["flashcard"]["question"],
            "correct_answer": correct_ans,
            "user_answer": user_ans,
            "result": "correct" if is_correct else "incorrect"
        }]
    }

# 5. Show Feedback
def show_feedback(state: StudyState):
    print(state["feedback"])
    return {}

# 6. Build Graph
graph = StateGraph(StudyState)

graph.add_node("generate", generate_flashcard)
graph.add_node("ask", ask_question)
graph.add_node("evaluate", evaluate_answer)
graph.add_node("feedback", show_feedback)

graph.set_entry_point("generate")
graph.add_edge("generate", "ask")
graph.add_edge("ask", "evaluate")
graph.add_edge("evaluate", "feedback")
graph.add_edge("feedback", END)

# 7. Run
if __name__ == "__main__":
    app = graph.compile()

    state = {
        "topic": input("Enter topic: "),
        "flashcard": {},
        "user_answer": "",
        "feedback": "",
        "progress": []
    }

    while True:
        state["flashcard"] = {}
        state["user_answer"] = ""
        state["feedback"] = ""

        state = app.invoke(state)

        cont = input("\n👉 Continue? (yes/no): ").strip().lower()
        if cont != "yes":
            break

    print("\n📊 Final Progress:")
    for p in state["progress"]:
        print(p)


RAW LLM OUTPUT:
 Question: What is the basic structural and functional unit of life?
Answer: Cell

📚 Question: What is the basic structural and functional unit of life?
✅ Correct!

📊 Final Progress:
{'question': 'What is the basic structural and functional unit of life?', 'correct_answer': 'cell', 'user_answer': 'cell', 'result': 'correct'}
